# DataLoader As A Queueing System

## Where the input work lives, why heatmaps matter, and when DDP changes the answer

A training input pipeline is a **parallel queueing system**. The job is to keep GPUs fed by balancing file reads, decode, transforms, batching, queues, H2D transfer, and rank synchronization.


In [ ]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from apps.story.dataloader_figures import FIGURE_GROUPS, FIGURE_FILENAMES, public_figure_url

pd.options.display.max_colwidth = 120
pio.renderers.default = "notebook_connected"


def find_repo_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "data/generated").exists() and (candidate / "pyproject.toml").exists():
            return candidate
    return Path.cwd().resolve()

ROOT = find_repo_root()
DATA = ROOT / "data/generated"

COLORS = {
    "cpu": "#2f6f73",
    "dali": "#b45f06",
    "uint8": "#4c78a8",
    "fp16": "#f58518",
    "synthetic": "#7a5195",
    "line": "#5f6368",
    "storage": "#6b7280",
}


def csv(path):
    return pd.read_csv(DATA / path)


def num(value):
    if pd.isna(value):
        return None
    if isinstance(value, (int, float)):
        return float(value)
    text = str(value).strip().replace(",", "").replace("%", "").replace("x", "")
    return float(text)


def with_numbers(df, columns):
    out = df.copy()
    for column in columns:
        out[column] = out[column].map(num)
    return out


def clean_layout(fig, title, height=520):
    fig.update_layout(
        title=title,
        height=height,
        template="plotly_white",
        margin=dict(l=55, r=35, t=80, b=60),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        font=dict(size=15),
    )
    return fig


def pipeline_sankey():
    nodes = [
        "Storage", "CPU memory / page cache", "CPU workers", "CPU batch",
        "H2D copy", "GPU memory", "Benchmark loop", "DALI CPU reader",
        "DALI mixed decode", "DALI GPU ops", "Offline prep already paid",
        "NumPy tensor read", "Synthetic GPU generator",
    ]
    node_color = [
        "#9ca3af", "#cbd5e1", "#2f6f73", "#93c5fd", "#64748b", "#f59e0b", "#111827",
        "#b45f06", "#d97706", "#f59e0b", "#8b5cf6", "#4c78a8", "#7a5195",
    ]
    flows = {
        "PyTorch CPU JPEG": ([0, 1, 2, 3, 4, 5], [1, 2, 3, 4, 5, 6], [3, 3, 3, 3, 3, 3]),
        "DALI JPEG": ([0, 7, 8, 9, 5], [7, 8, 9, 5, 6], [3, 3, 3, 3, 3]),
        "NumPy uint8 prepared": ([10, 0, 11, 2, 3, 4, 5], [0, 11, 2, 3, 4, 5, 6], [2, 2, 2, 2, 2, 2, 2]),
        "NumPy fp16 ceiling": ([10, 0, 11, 3, 4, 5], [0, 11, 3, 4, 5, 6], [2, 2, 2, 2, 2, 2]),
        "Synthetic GPU ceiling": ([12, 5], [5, 6], [3, 3]),
    }
    fig = go.Figure()
    for index, (name, (source, target, value)) in enumerate(flows.items()):
        fig.add_trace(go.Sankey(
            visible=index == 0,
            arrangement="snap",
            node=dict(label=nodes, pad=16, thickness=18, color=node_color),
            link=dict(source=source, target=target, value=value, color="rgba(47,111,115,0.28)"),
            name=name,
        ))
    buttons = []
    for index, name in enumerate(flows):
        visible = [False] * len(flows)
        visible[index] = True
        buttons.append(dict(label=name, method="update", args=[{"visible": visible}, {"title": f"Where the input work lives: {name}"}]))
    fig.update_layout(updatemenus=[dict(buttons=buttons, direction="down", x=0, y=1.18, xanchor="left")])
    return clean_layout(fig, "Where the input work lives: PyTorch CPU JPEG", height=560)


def worker_scan_fig():
    frames = []
    for platform, path in [
        ("B200", "dataloader/b200-single-node-replicated-table-2.csv"),
        ("RTX Pro 6000", "dataloader/rtx-single-node-replicated-table-2.csv"),
    ]:
        df = with_numbers(csv(path), ["samples_s", "retained_imbalance", "retained_jitter"])
        df["platform"] = platform
        frames.append(df)
    data = pd.concat(frames, ignore_index=True)
    combos = [(p, b) for p in data.platform.unique() for b in sorted(data.loc[data.platform == p, "batch"].unique())]
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    for i, (platform, batch) in enumerate(combos):
        sub = data[(data.platform == platform) & (data.batch == batch)].sort_values("workers")
        visible = i == 0
        fig.add_trace(go.Scatter(x=sub.workers, y=sub.samples_s, mode="lines+markers", name="samples/s", visible=visible, marker_color=COLORS["cpu"]), secondary_y=False)
        fig.add_trace(go.Scatter(x=sub.workers, y=sub.retained_imbalance, mode="lines+markers", name="rank imbalance %", visible=visible, marker_color=COLORS["line"], line=dict(dash="dot")), secondary_y=True)
    buttons = []
    for i, (platform, batch) in enumerate(combos):
        visible = [False] * (2 * len(combos))
        visible[2*i] = True
        visible[2*i + 1] = True
        buttons.append(dict(label=f"{platform}, batch {batch}", method="update", args=[{"visible": visible}, {"title": f"Workers feed eight ranks: {platform}, batch {batch}"}]))
    fig.update_layout(updatemenus=[dict(buttons=buttons, direction="down", x=0, y=1.18, xanchor="left")])
    fig.update_xaxes(title_text="num_workers per rank")
    fig.update_yaxes(title_text="samples/s", secondary_y=False)
    fig.update_yaxes(title_text="retained rank imbalance %", secondary_y=True)
    return clean_layout(fig, "Workers feed eight ranks: B200, batch 512", height=540)


def backend_crossover_fig():
    df = with_numbers(csv("dataloader/optimized-backend-crossover-table-2.csv"), ["cpu_samples_s", "dali_samples_s", "dali_cpu"])
    platforms = list(df.platform.unique())
    fig = go.Figure()
    for i, platform in enumerate(platforms):
        sub = df[df.platform == platform]
        visible = i == 0
        hover_cpu = [f"{row.endpoint}<br>{row.best_cpu_config}<br>{row.cpu_samples_s:,.0f} samples/s" for row in sub.itertuples()]
        hover_dali = [f"{row.endpoint}<br>{row.best_dali_config}<br>{row.dali_samples_s:,.0f} samples/s<br>DALI/CPU {row.dali_cpu:.2f}x" for row in sub.itertuples()]
        fig.add_bar(x=sub.endpoint, y=sub.cpu_samples_s, name="PyTorch CPU", marker_color=COLORS["cpu"], visible=visible, hovertext=hover_cpu, hoverinfo="text")
        fig.add_bar(x=sub.endpoint, y=sub.dali_samples_s, name="DALI", marker_color=COLORS["dali"], visible=visible, hovertext=hover_dali, hoverinfo="text")
    buttons = []
    for i, platform in enumerate(platforms):
        visible = [False] * (2 * len(platforms))
        visible[2*i] = True
        visible[2*i + 1] = True
        buttons.append(dict(label=platform, method="update", args=[{"visible": visible}, {"title": f"Backend choice changes with representation: {platform}"}]))
    fig.update_layout(barmode="group", updatemenus=[dict(buttons=buttons, direction="down", x=0, y=1.18, xanchor="left")])
    fig.update_yaxes(title_text="DataLoader samples/s")
    return clean_layout(fig, "Backend choice changes with representation: B200", height=540)


def prepared_economics_fig():
    throughput = with_numbers(csv("dataloader/prepared-input-ceilings-table-2.csv"), ["uint8_samples_s", "fp16_samples_s", "fp16_uint8"])
    storage = with_numbers(csv("dataloader/prepared-input-ceilings-table-3.csv"), ["uint8_dataset_gib", "fp16_dataset_gib"])
    platforms = list(throughput.platform.unique())
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    for i, platform in enumerate(platforms):
        sub = throughput[throughput.platform == platform].sort_values("size")
        visible = i == 0
        fig.add_trace(go.Scatter(x=sub["size"], y=sub.uint8_samples_s, mode="lines+markers", name=f"{platform} uint8 samples/s", visible=visible, marker_color=COLORS["uint8"]), secondary_y=False)
        fig.add_trace(go.Scatter(x=sub["size"], y=sub.fp16_samples_s, mode="lines+markers", name=f"{platform} fp16 samples/s", visible=visible, marker_color=COLORS["fp16"]), secondary_y=False)
    fig.add_trace(go.Scatter(x=storage["size"], y=storage.uint8_dataset_gib, mode="lines+markers", name="uint8 GiB", marker_color=COLORS["storage"], line=dict(dash="dash"), visible=True), secondary_y=True)
    fig.add_trace(go.Scatter(x=storage["size"], y=storage.fp16_dataset_gib, mode="lines+markers", name="fp16 GiB", marker_color="#374151", line=dict(dash="dot"), visible=True), secondary_y=True)
    buttons = []
    trace_count = 2 * len(platforms) + 2
    for i, platform in enumerate(platforms):
        visible = [False] * trace_count
        visible[2*i] = True
        visible[2*i + 1] = True
        visible[-2] = True
        visible[-1] = True
        buttons.append(dict(label=platform, method="update", args=[{"visible": visible}, {"title": f"Prepared inputs: pay storage/precompute once, {platform}"}]))
    fig.update_layout(updatemenus=[dict(buttons=buttons, direction="down", x=0, y=1.18, xanchor="left")])
    fig.update_xaxes(title_text="prepared image/tensor size")
    fig.update_yaxes(title_text="DataLoader samples/s", secondary_y=False)
    fig.update_yaxes(title_text="derived dataset GiB", secondary_y=True)
    return clean_layout(fig, "Prepared inputs: pay storage/precompute once, B200", height=560)


def ddp_truth_fig():
    one = with_numbers(csv("ddp/dataloader-candidate-followup-table-2.csv"), ["pytorch_cpu_images_s", "dali_images_s", "dali_pytorch"])
    scale = with_numbers(csv("ddp/multinode-scale-validation-table-2.csv"), ["1_node_img_s", "2_node_img_s", "4_node_img_s"])
    platforms = list(one.platform.unique())
    fig = make_subplots(rows=1, cols=2, subplot_titles=("One-node candidate validation", "Multi-node scale check"))
    for i, platform in enumerate(platforms):
        visible = i == 0
        sub = one[one.platform == platform]
        fig.add_trace(go.Bar(x=sub.endpoint, y=sub.pytorch_cpu_images_s, name="PyTorch CPU images/s", marker_color=COLORS["cpu"], visible=visible), row=1, col=1)
        fig.add_trace(go.Bar(x=sub.endpoint, y=sub.dali_images_s, name="DALI images/s", marker_color=COLORS["dali"], visible=visible), row=1, col=1)
        ss = scale[scale.platform == platform]
        for _, row in ss.iterrows():
            fig.add_trace(go.Scatter(x=[1,2,4], y=[row["1_node_img_s"], row["2_node_img_s"], row["4_node_img_s"]], mode="lines+markers", name=row["candidate"], visible=visible), row=1, col=2)
    traces_per_platform = 4
    buttons = []
    for i, platform in enumerate(platforms):
        visible = [False] * (traces_per_platform * len(platforms))
        for t in range(traces_per_platform):
            visible[i * traces_per_platform + t] = True
        buttons.append(dict(label=platform, method="update", args=[{"visible": visible}, {"title": f"DDP is the training truth: {platform}"}]))
    fig.update_layout(barmode="group", updatemenus=[dict(buttons=buttons, direction="down", x=0, y=1.18, xanchor="left")])
    fig.update_xaxes(title_text="endpoint", row=1, col=1)
    fig.update_xaxes(title_text="nodes", row=1, col=2)
    fig.update_yaxes(title_text="images/s", row=1, col=1)
    fig.update_yaxes(title_text="images/s", row=1, col=2)
    return clean_layout(fig, "DDP is the training truth: B200", height=560)


def input_ceiling_fig():
    df = with_numbers(csv("ddp/input-ceilings-table-2.csv"), ["images_s"])
    platforms = list(df.platform.unique())
    sizes = sorted(df["size"].unique())
    combos = [(p, s) for p in platforms for s in sizes]
    fig = go.Figure()
    for i, (platform, size) in enumerate(combos):
        sub = df[(df.platform == platform) & (df["size"] == size)]
        fig.add_bar(x=sub.input, y=sub.images_s, name=f"{platform} {size}", visible=i == 0, marker_color=[COLORS["uint8"], COLORS["fp16"], COLORS["synthetic"]])
    buttons = []
    for i, (platform, size) in enumerate(combos):
        visible = [False] * len(combos)
        visible[i] = True
        buttons.append(dict(label=f"{platform}, {size}", method="update", args=[{"visible": visible}, {"title": f"Synthetic GPU input removes the real input path: {platform}, {size}"}]))
    fig.update_layout(updatemenus=[dict(buttons=buttons, direction="down", x=0, y=1.18, xanchor="left")])
    fig.update_yaxes(title_text="DDP images/s")
    return clean_layout(fig, "Synthetic GPU input removes the real input path: B200, 224", height=520)


<style>
.aicr-figure-grid {
  display: grid;
  grid-template-columns: repeat(2, minmax(0, 1fr));
  gap: 14px;
  align-items: start;
}
.aicr-figure-single {
  display: flex;
  justify-content: center;
}
.aicr-figure-grid figure,
.aicr-figure-single figure {
  margin: 0;
  text-align: center;
}
.aicr-figure-grid img,
.aicr-figure-single img {
  width: 100%;
  max-height: 520px;
  object-fit: contain;
  border: 1px solid #d7dde5;
  background: #fff;
}
.aicr-figure-single img {
  max-width: 92%;
  max-height: 610px;
}
.aicr-figure-grid figcaption,
.aicr-figure-single figcaption {
  font-size: 0.45em;
  color: #4b5563;
  margin-top: 4px;
  overflow-wrap: anywhere;
}
</style>


## The Teaching Arc

1. **Where work lives**: CPU JPEG, DALI JPEG, prepared inputs, synthetic ceilings.
2. **How knobs control parallelism**: workers, prefetch, batch, H2D staging, DALI threads/queue depth, ranks and nodes.
3. **Why heatmaps are the story**: plateaus, hidden imbalance, candidate frontiers, and scale survival.
4. **Why representation changes the answer**: `canonical-224`, `derived-jpeg-1024`, input lab, prepared ceilings.
5. **Why DDP is the truth test**: DataLoader finds candidates; training validates them.


## Act 1: Where Work Lives

The same image can pay different runtime tolls depending on the path.

- PyTorch CPU path: storage -> CPU workers -> CPU decode/transform -> H2D -> GPU.
- DALI path: storage -> DALI reader -> mixed decode/GPU operators -> GPU.
- Prepared path: move decode and normalization offline.
- Synthetic path: remove input entirely as a ceiling.

The selector below is not the story endpoint. It is the map for reading the plots that follow.


In [ ]:
pipeline_sankey().show()


## Act 2: Knobs Control Parallelism

| Knob | Parallel work it controls | What the plots teach |
| --- | --- | --- |
| `num_workers` | CPU file read, JPEG decode, transform, collation. | Throughput rises, then plateaus or destabilizes. |
| `prefetch_factor` | Queue depth and overlap. | More queued work can hide latency, or just add memory pressure. |
| `batch_size` | Amortization and memory footprint. | Bigger batches help until they expose memory/rank costs. |
| `pin_memory` | H2D staging. | Only matters if host-to-device transfer is in the limiting path. |
| DALI threads / queue depth | DALI pipeline parallelism and overlap. | Useful for large images; not free for ordinary ImageNet. |
| rank and node count | Distributed balance and synchronization pressure. | A fast local row can be a bad distributed candidate. |


## Diagnostic Map: If You See X, Inspect Y First (Not A Recipe)

| Symptom | First inspection | Why |
| --- | --- | --- |
| Throughput stops rising across workers. | Worker plateau and CPU-side decode pressure. | More workers no longer add useful parallelism. |
| High throughput but high imbalance. | Candidate scatter and imbalance heatmaps. | Slow ranks set distributed pace. |
| DALI helps only for larger inputs. | Representation and image size. | Pipeline overhead must be amortized. |
| H2D rate far above `samples/s`. | Decode/transform/storage before H2D. | Transfer is probably not the limiting toll. |
| Prepared fp16 looks much faster. | Storage and preprocessing debt. | Work moved offline, not eliminated for free. |
| DataLoader wins but training does not. | DDP validation and synthetic ceiling. | Model and sync can dominate. |


## Heatmaps Are The Story

**Throughput heatmaps** show the plateau: where more workers, batch, or prefetch stop increasing useful work.

**Imbalance heatmaps** show the hidden distributed cost: a visually fast row can be a bad candidate if queueing leaves some ranks behind.

**Candidate scatter plots** show the frontier: the interesting rows are high throughput with acceptable imbalance, not simply the highest dot.

Scale plots test whether the chosen balance survives more nodes.


## Interactive Lens: Workers Feed Eight Ranks

Pick platform and batch, then look at how throughput and rank imbalance move together.

This curve is the simplified reader for the one-node heatmaps: it teaches the plateau before the full matrix appears.


In [ ]:
worker_scan_fig().show()


## Plot Family: Single-GPU Surface

The first surface shows the local throughput plateau before ranks enter the story.

Read every plot as a queueing/balance question: what knob increased useful overlap, where did throughput plateau, and where did imbalance make a row less attractive?


## Single-GPU Throughput Surfaces

<div class="aicr-figure-grid">
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-single-gpu-throughput-b200-2026-05-12.png" alt="dataloader-single-gpu-throughput-b200-2026-05-12.png" />
    <figcaption>dataloader-single-gpu-throughput-b200-2026-05-12.png</figcaption>
  </figure>
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-single-gpu-throughput-rtxpro6000-2026-05-12.png" alt="dataloader-single-gpu-throughput-rtxpro6000-2026-05-12.png" />
    <figcaption>dataloader-single-gpu-throughput-rtxpro6000-2026-05-12.png</figcaption>
  </figure>
</div>


## Plot Family: One-Node Replicated Rank Pressure

The one-node plots show how worker count, batch, and prefetch feed eight ranks; imbalance is the hidden cost.

Read every plot as a queueing/balance question: what knob increased useful overlap, where did throughput plateau, and where did imbalance make a row less attractive?


## B200 One-Node Throughput And Imbalance

<div class="aicr-figure-grid">
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-one-node-replicated-throughput-matrix-b200-2026-05-12.png" alt="dataloader-one-node-replicated-throughput-matrix-b200-2026-05-12.png" />
    <figcaption>dataloader-one-node-replicated-throughput-matrix-b200-2026-05-12.png</figcaption>
  </figure>
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-one-node-replicated-imbalance-matrix-b200-2026-05-12.png" alt="dataloader-one-node-replicated-imbalance-matrix-b200-2026-05-12.png" />
    <figcaption>dataloader-one-node-replicated-imbalance-matrix-b200-2026-05-12.png</figcaption>
  </figure>
</div>


## B200 One-Node Candidate Frontier


<div class="aicr-figure-single">
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-one-node-replicated-candidate-scatter-b200-2026-05-12.png" alt="dataloader-one-node-replicated-candidate-scatter-b200-2026-05-12.png" />
    <figcaption>dataloader-one-node-replicated-candidate-scatter-b200-2026-05-12.png</figcaption>
  </figure>
</div>


## RTX One-Node Throughput And Imbalance

<div class="aicr-figure-grid">
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-one-node-replicated-throughput-matrix-rtxpro6000-2026-05-12.png" alt="dataloader-one-node-replicated-throughput-matrix-rtxpro6000-2026-05-12.png" />
    <figcaption>dataloader-one-node-replicated-throughput-matrix-rtxpro6000-2026-05-12.png</figcaption>
  </figure>
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-one-node-replicated-imbalance-matrix-rtxpro6000-2026-05-12.png" alt="dataloader-one-node-replicated-imbalance-matrix-rtxpro6000-2026-05-12.png" />
    <figcaption>dataloader-one-node-replicated-imbalance-matrix-rtxpro6000-2026-05-12.png</figcaption>
  </figure>
</div>


## RTX One-Node Candidate Frontier


<div class="aicr-figure-single">
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-one-node-replicated-candidate-scatter-rtxpro6000-2026-05-12.png" alt="dataloader-one-node-replicated-candidate-scatter-rtxpro6000-2026-05-12.png" />
    <figcaption>dataloader-one-node-replicated-candidate-scatter-rtxpro6000-2026-05-12.png</figcaption>
  </figure>
</div>


## RTX Batch/Prefetch Refinement Heatmaps

<div class="aicr-figure-grid">
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-one-node-replicated-batch-prefetch-throughput-matrix-rtxpro6000-2026-05-12.png" alt="dataloader-one-node-replicated-batch-prefetch-throughput-matrix-rtxpro6000-2026-05-12.png" />
    <figcaption>dataloader-one-node-replicated-batch-prefetch-throughput-matrix-rtxpro6000-2026-05-12.png</figcaption>
  </figure>
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-one-node-replicated-batch-prefetch-imbalance-matrix-rtxpro6000-2026-05-12.png" alt="dataloader-one-node-replicated-batch-prefetch-imbalance-matrix-rtxpro6000-2026-05-12.png" />
    <figcaption>dataloader-one-node-replicated-batch-prefetch-imbalance-matrix-rtxpro6000-2026-05-12.png</figcaption>
  </figure>
</div>


## RTX Batch/Prefetch Candidate Frontier


<div class="aicr-figure-single">
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-one-node-replicated-batch-prefetch-candidate-scatter-rtxpro6000-2026-05-12.png" alt="dataloader-one-node-replicated-batch-prefetch-candidate-scatter-rtxpro6000-2026-05-12.png" />
    <figcaption>dataloader-one-node-replicated-batch-prefetch-candidate-scatter-rtxpro6000-2026-05-12.png</figcaption>
  </figure>
</div>


## Plot Family: Multi-Node Parameter Selection

The multi-node plots show where a high-throughput local row becomes a distributed balance problem.

Read every plot as a queueing/balance question: what knob increased useful overlap, where did throughput plateau, and where did imbalance make a row less attractive?


## B200 First-Stage Multi-Node Heatmaps

<div class="aicr-figure-grid">
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-b200-multinode-first-stage-throughput-matrix-2026-05-13.png" alt="dataloader-b200-multinode-first-stage-throughput-matrix-2026-05-13.png" />
    <figcaption>dataloader-b200-multinode-first-stage-throughput-matrix-2026-05-13.png</figcaption>
  </figure>
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-b200-multinode-first-stage-imbalance-matrix-2026-05-13.png" alt="dataloader-b200-multinode-first-stage-imbalance-matrix-2026-05-13.png" />
    <figcaption>dataloader-b200-multinode-first-stage-imbalance-matrix-2026-05-13.png</figcaption>
  </figure>
</div>


## B200 Per-Node And Candidate Frontier

<div class="aicr-figure-grid">
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-b200-multinode-first-stage-throughput-per-node-matrix-2026-05-13.png" alt="dataloader-b200-multinode-first-stage-throughput-per-node-matrix-2026-05-13.png" />
    <figcaption>dataloader-b200-multinode-first-stage-throughput-per-node-matrix-2026-05-13.png</figcaption>
  </figure>
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-b200-multinode-followup-candidate-scatter-2026-05-13.png" alt="dataloader-b200-multinode-followup-candidate-scatter-2026-05-13.png" />
    <figcaption>dataloader-b200-multinode-followup-candidate-scatter-2026-05-13.png</figcaption>
  </figure>
</div>


## B200 Multi-Node Samples Per Node


<div class="aicr-figure-single">
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-b200-multinode-samples-per-node-2026-05-13.png" alt="dataloader-b200-multinode-samples-per-node-2026-05-13.png" />
    <figcaption>dataloader-b200-multinode-samples-per-node-2026-05-13.png</figcaption>
  </figure>
</div>


## RTX First-Stage Multi-Node Heatmaps

<div class="aicr-figure-grid">
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-rtx-multinode-throughput-matrix-2026-05-12-13.png" alt="dataloader-rtx-multinode-throughput-matrix-2026-05-12-13.png" />
    <figcaption>dataloader-rtx-multinode-throughput-matrix-2026-05-12-13.png</figcaption>
  </figure>
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-rtx-multinode-imbalance-matrix-2026-05-12-13.png" alt="dataloader-rtx-multinode-imbalance-matrix-2026-05-12-13.png" />
    <figcaption>dataloader-rtx-multinode-imbalance-matrix-2026-05-12-13.png</figcaption>
  </figure>
</div>


## RTX Per-Node And Candidate Frontier

<div class="aicr-figure-grid">
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-rtx-multinode-throughput-per-node-matrix-2026-05-12-13.png" alt="dataloader-rtx-multinode-throughput-per-node-matrix-2026-05-12-13.png" />
    <figcaption>dataloader-rtx-multinode-throughput-per-node-matrix-2026-05-12-13.png</figcaption>
  </figure>
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-rtx-multinode-candidate-scatter-2026-05-12-13.png" alt="dataloader-rtx-multinode-candidate-scatter-2026-05-12-13.png" />
    <figcaption>dataloader-rtx-multinode-candidate-scatter-2026-05-12-13.png</figcaption>
  </figure>
</div>


## RTX Follow-Up Multi-Node Heatmaps

<div class="aicr-figure-grid">
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-rtx-multinode-imbalance-matrix-2026-05-13.png" alt="dataloader-rtx-multinode-imbalance-matrix-2026-05-13.png" />
    <figcaption>dataloader-rtx-multinode-imbalance-matrix-2026-05-13.png</figcaption>
  </figure>
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-rtx-multinode-candidate-scatter-2026-05-13.png" alt="dataloader-rtx-multinode-candidate-scatter-2026-05-13.png" />
    <figcaption>dataloader-rtx-multinode-candidate-scatter-2026-05-13.png</figcaption>
  </figure>
</div>


## RTX Multi-Node Samples Per Node


<div class="aicr-figure-single">
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-rtx-multinode-samples-per-node-2026-05-13.png" alt="dataloader-rtx-multinode-samples-per-node-2026-05-13.png" />
    <figcaption>dataloader-rtx-multinode-samples-per-node-2026-05-13.png</figcaption>
  </figure>
</div>


## Plot Family: Scale Probe

Scale plots test whether the chosen balance survives more nodes.

Read every plot as a queueing/balance question: what knob increased useful overlap, where did throughput plateau, and where did imbalance make a row less attractive?


## B200 8/16-Node Heatmaps

<div class="aicr-figure-grid">
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-b200-8-16node-throughput-matrix-2026-05-14.png" alt="dataloader-b200-8-16node-throughput-matrix-2026-05-14.png" />
    <figcaption>dataloader-b200-8-16node-throughput-matrix-2026-05-14.png</figcaption>
  </figure>
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-b200-8-16node-imbalance-matrix-2026-05-14.png" alt="dataloader-b200-8-16node-imbalance-matrix-2026-05-14.png" />
    <figcaption>dataloader-b200-8-16node-imbalance-matrix-2026-05-14.png</figcaption>
  </figure>
</div>


## B200 2/4/8/16 Scale

<div class="aicr-figure-grid">
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-b200-multinode-samples-per-node-2-4-8-16-2026-05-14.png" alt="dataloader-b200-multinode-samples-per-node-2-4-8-16-2026-05-14.png" />
    <figcaption>dataloader-b200-multinode-samples-per-node-2-4-8-16-2026-05-14.png</figcaption>
  </figure>
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-b200-multinode-imbalance-2-4-8-16-2026-05-14.png" alt="dataloader-b200-multinode-imbalance-2-4-8-16-2026-05-14.png" />
    <figcaption>dataloader-b200-multinode-imbalance-2-4-8-16-2026-05-14.png</figcaption>
  </figure>
</div>


## B200 Scale Candidate Frontier


<div class="aicr-figure-single">
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-b200-scale-probe-candidate-scatter-2026-05-14.png" alt="dataloader-b200-scale-probe-candidate-scatter-2026-05-14.png" />
    <figcaption>dataloader-b200-scale-probe-candidate-scatter-2026-05-14.png</figcaption>
  </figure>
</div>


## RTX Eight-Node Heatmaps

<div class="aicr-figure-grid">
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-rtx-8node-throughput-matrix-2026-05-13.png" alt="dataloader-rtx-8node-throughput-matrix-2026-05-13.png" />
    <figcaption>dataloader-rtx-8node-throughput-matrix-2026-05-13.png</figcaption>
  </figure>
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-rtx-8node-imbalance-matrix-2026-05-13.png" alt="dataloader-rtx-8node-imbalance-matrix-2026-05-13.png" />
    <figcaption>dataloader-rtx-8node-imbalance-matrix-2026-05-13.png</figcaption>
  </figure>
</div>


## RTX Eight-Node Candidate Frontier


<div class="aicr-figure-single">
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-rtx-8node-candidate-scatter-2026-05-13.png" alt="dataloader-rtx-8node-candidate-scatter-2026-05-13.png" />
    <figcaption>dataloader-rtx-8node-candidate-scatter-2026-05-13.png</figcaption>
  </figure>
</div>


## RTX 2/4/8 Scale

<div class="aicr-figure-grid">
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-rtx-multinode-samples-per-node-2-4-8-2026-05-13.png" alt="dataloader-rtx-multinode-samples-per-node-2-4-8-2026-05-13.png" />
    <figcaption>dataloader-rtx-multinode-samples-per-node-2-4-8-2026-05-13.png</figcaption>
  </figure>
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-rtx-multinode-imbalance-2-4-8-2026-05-13.png" alt="dataloader-rtx-multinode-imbalance-2-4-8-2026-05-13.png" />
    <figcaption>dataloader-rtx-multinode-imbalance-2-4-8-2026-05-13.png</figcaption>
  </figure>
</div>


## Act 4: Representation Changes The Answer

The backend decision is not universal; the input representation changes which toll dominates.

- `canonical-224`: CPU path is strong because ordinary decode/resize pressure is not large enough to amortize DALI overhead.
- `derived-jpeg-1024`: DALI wins because decode and image-processing pressure dominate.
- Prepared inputs teach “pay once offline” versus “pay every run online.”
- Synthetic input removes the real input path and is only a ceiling/control.


## Interactive Lens: Backend Crossover

Pick platform, then compare ordinary ImageNet against large derived JPEG.

This is the compact reader for the DALI and crossover plot family: representation changes the experiment.


In [ ]:
backend_crossover_fig().show()


## Plot Family: DALI And Backend Crossover

These plots show why representation and image size change whether DALI amortizes its pipeline overhead.

Keep the taxonomy strict: canonical ImageNet, derived JPEG, and input-lab rows answer different questions.


## DALI On Standard ImageNet

<div class="aicr-figure-grid">
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-dali-standard-imagenet-b200-2026-05-20-top-configs.png" alt="dataloader-dali-standard-imagenet-b200-2026-05-20-top-configs.png" />
    <figcaption>dataloader-dali-standard-imagenet-b200-2026-05-20-top-configs.png</figcaption>
  </figure>
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-dali-standard-imagenet-rtxpro6000-2026-05-20-top-configs.png" alt="dataloader-dali-standard-imagenet-rtxpro6000-2026-05-20-top-configs.png" />
    <figcaption>dataloader-dali-standard-imagenet-rtxpro6000-2026-05-20-top-configs.png</figcaption>
  </figure>
</div>


## CPU 1024 JPEG Tuning

<div class="aicr-figure-grid">
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-cpu-1024-optimization-b200-2026-05-20-top-configs.png" alt="dataloader-cpu-1024-optimization-b200-2026-05-20-top-configs.png" />
    <figcaption>dataloader-cpu-1024-optimization-b200-2026-05-20-top-configs.png</figcaption>
  </figure>
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-cpu-1024-optimization-rtxpro6000-2026-05-20-top-configs.png" alt="dataloader-cpu-1024-optimization-rtxpro6000-2026-05-20-top-configs.png" />
    <figcaption>dataloader-cpu-1024-optimization-rtxpro6000-2026-05-20-top-configs.png</figcaption>
  </figure>
</div>


## DALI 1024 JPEG Tuning

<div class="aicr-figure-grid">
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-dali-1024-optimization-b200-2026-05-20-top-configs.png" alt="dataloader-dali-1024-optimization-b200-2026-05-20-top-configs.png" />
    <figcaption>dataloader-dali-1024-optimization-b200-2026-05-20-top-configs.png</figcaption>
  </figure>
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-dali-1024-optimization-rtxpro6000-2026-05-20-top-configs.png" alt="dataloader-dali-1024-optimization-rtxpro6000-2026-05-20-top-configs.png" />
    <figcaption>dataloader-dali-1024-optimization-rtxpro6000-2026-05-20-top-configs.png</figcaption>
  </figure>
</div>


## Optimized Backend Crossover

<div class="aicr-figure-grid">
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-optimized-224-1024-backend-crossover-2026-05-20.png" alt="dataloader-optimized-224-1024-backend-crossover-2026-05-20.png" />
    <figcaption>dataloader-optimized-224-1024-backend-crossover-2026-05-20.png</figcaption>
  </figure>
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-backend-dali-crossover-100x500-2026-05-23.png" alt="dataloader-backend-dali-crossover-100x500-2026-05-23.png" />
    <figcaption>dataloader-backend-dali-crossover-100x500-2026-05-23.png</figcaption>
  </figure>
</div>


## Derived 224 Context


<div class="aicr-figure-single">
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-backend-dali-crossover-224-context-2026-05-24.png" alt="dataloader-backend-dali-crossover-224-context-2026-05-24.png" />
    <figcaption>dataloader-backend-dali-crossover-224-context-2026-05-24.png</figcaption>
  </figure>
</div>


## Plot Family: Input Lab And Representation

Input-lab plots show how image size and representation change the mechanism, not just the winning backend.

Keep the taxonomy strict: canonical ImageNet, derived JPEG, and input-lab rows answer different questions.


## Input-Lab Throughput By Platform

<div class="aicr-figure-grid">
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-input-lab-throughput-b200-2026-05-19.png" alt="dataloader-input-lab-throughput-b200-2026-05-19.png" />
    <figcaption>dataloader-input-lab-throughput-b200-2026-05-19.png</figcaption>
  </figure>
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-input-lab-throughput-rtxpro6000-2026-05-19.png" alt="dataloader-input-lab-throughput-rtxpro6000-2026-05-19.png" />
    <figcaption>dataloader-input-lab-throughput-rtxpro6000-2026-05-19.png</figcaption>
  </figure>
</div>


## Input-Lab Image Size Effect

<div class="aicr-figure-grid">
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-input-lab-image-size-b200-2026-05-19.png" alt="dataloader-input-lab-image-size-b200-2026-05-19.png" />
    <figcaption>dataloader-input-lab-image-size-b200-2026-05-19.png</figcaption>
  </figure>
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-input-lab-image-size-rtxpro6000-2026-05-19.png" alt="dataloader-input-lab-image-size-rtxpro6000-2026-05-19.png" />
    <figcaption>dataloader-input-lab-image-size-rtxpro6000-2026-05-19.png</figcaption>
  </figure>
</div>


## Input-Lab Speedup Versus Original

<div class="aicr-figure-grid">
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-input-lab-speedup-original-b200-2026-05-19.png" alt="dataloader-input-lab-speedup-original-b200-2026-05-19.png" />
    <figcaption>dataloader-input-lab-speedup-original-b200-2026-05-19.png</figcaption>
  </figure>
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-input-lab-speedup-original-rtxpro6000-2026-05-19.png" alt="dataloader-input-lab-speedup-original-rtxpro6000-2026-05-19.png" />
    <figcaption>dataloader-input-lab-speedup-original-rtxpro6000-2026-05-19.png</figcaption>
  </figure>
</div>


## Input-Lab Same-Size Speedup

<div class="aicr-figure-grid">
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-input-lab-speedup-same-size-b200-2026-05-19.png" alt="dataloader-input-lab-speedup-same-size-b200-2026-05-19.png" />
    <figcaption>dataloader-input-lab-speedup-same-size-b200-2026-05-19.png</figcaption>
  </figure>
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-input-lab-speedup-same-size-rtxpro6000-2026-05-19.png" alt="dataloader-input-lab-speedup-same-size-rtxpro6000-2026-05-19.png" />
    <figcaption>dataloader-input-lab-speedup-same-size-rtxpro6000-2026-05-19.png</figcaption>
  </figure>
</div>


## Interactive Lens: Prepared-Input Economics

Pick platform, then compare throughput against storage footprint.

Prepared inputs are **ceiling and economics evidence**, not a default recipe. They show what happens when decode and normalization are moved out of the online path and fixed into stored tensors.


In [ ]:
prepared_economics_fig().show()


## Plot Family: Prepared-Input Ceilings

Prepared-input plots teach pay once offline versus pay every run online.

Read these as ceiling evidence and storage/precompute economics, not as a universal training recommendation.


## Prepared-Input Throughput And Read Rate

<div class="aicr-figure-grid">
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-prepared-input-finalist-throughput-2026-05-20.png" alt="dataloader-prepared-input-finalist-throughput-2026-05-20.png" />
    <figcaption>dataloader-prepared-input-finalist-throughput-2026-05-20.png</figcaption>
  </figure>
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-prepared-input-finalist-read-rate-2026-05-20.png" alt="dataloader-prepared-input-finalist-read-rate-2026-05-20.png" />
    <figcaption>dataloader-prepared-input-finalist-read-rate-2026-05-20.png</figcaption>
  </figure>
</div>


## Prepared fp16 Speedup


<div class="aicr-figure-single">
  <figure>
    <img src="https://raw.githubusercontent.com/ohpcsim/aicr-public/b3e610f486b42d1363dbb0f2591a315116564498/Cambridge/aicr-bench/docs/modules/dataloader/studies/figures/dataloader-prepared-input-finalist-fp16-speedup-2026-05-20.png" alt="dataloader-prepared-input-finalist-fp16-speedup-2026-05-20.png" />
    <figcaption>dataloader-prepared-input-finalist-fp16-speedup-2026-05-20.png</figcaption>
  </figure>
</div>


## Normal DALI JPEG Is Not GDS

The standard DALI JPEG path in these studies is not a GPUDirect Storage result. These plots do not claim GDS throughput evidence for JPEG decode.

DALI NumPy CPU/GPU reader rows, when present, are prepared-tensor transport probes.

They are not evidence that normal DALI JPEG uses GDS, not evidence that JPEG decode improved through GDS, and not a general no-CPU-memory input recipe.

Do not publish a GDS performance claim without same-node `verify-gds` evidence and paired CPU-reader versus GPU/O_DIRECT-reader rows on the same prepared-tensor path.


## Act 5: DDP Is The Truth Test

DataLoader finds candidate input paths.

DDP tells whether those candidates still matter after model compute, backward pass, optimizer work, communication, and rank synchronization enter the loop.

This is why DataLoader-only throughput selects candidates; it does not prove training throughput.


## Interactive Lens: DDP Validation

Pick platform, then compare one-node candidate validation with a multi-node scale check.

DDP stays short in this deck: it is the training truth test for DataLoader candidates, not a separate module story.


In [ ]:
ddp_truth_fig().show()


## Synthetic GPU Input Is A Control

Synthetic GPU input removes the real input path.

It is useful because it shows a training-side ceiling: how fast the model loop can run when storage, decode, runtime preprocessing, and host-to-device input copy are removed.

Synthetic GPU input removes the real input path; it is not a dataset strategy.


In [ ]:
input_ceiling_fig().show()


## DDP Summary: DataLoader Nominates; DDP Decides

DataLoader heatmaps nominate input candidates by showing throughput plateaus, imbalance risk, and throughput/balance frontiers.

DDP decides whether a candidate matters in training after model compute, backward pass, optimizer work, communication, and rank synchronization are included.

Keep the evidence roles separate: `canonical-224` is the ordinary ImageNet starting point, `derived-jpeg-1024` stresses large compressed-image work, prepared inputs are ceiling/economics evidence, and synthetic GPU input is a compute-side control.


## Decision Framework

1. Start with `canonical-224` PyTorch CPU for ordinary ImageNet-like input.
2. Use heatmaps to find the plateau and imbalance plots to avoid bad distributed candidates.
3. Use scatter plots to pick the throughput/balance frontier, not just the highest row.
4. Try DALI when large compressed images or heavy image preprocessing dominate.
5. Try prepared inputs only when repeated runs justify storage and precompute cost.
6. Use synthetic ceilings to separate input bottlenecks from model-side ceilings.
7. Validate the chosen path in DDP before recommending a training strategy.


## What Not To Claim

Keep these guardrails in the main deck:

- Do not say DALI always wins.
- Do not say DataLoader-only wins are training wins.
- Do not recommend prepared fp16 as a general recipe.
- Do not claim the normal DALI JPEG path uses GDS.
- Do not compare full-resolution `1024` prepared or synthetic rows directly against JPEG rows that crop or resize to `224`.


## Reading Order After The Deck

Use the deck as a map, then read the public studies in this order:

1. DataLoader results summary.
2. Input pipeline reference and representation map.
3. Single-GPU and one-node heatmaps.
4. Multi-node parameter-selection and scale-probe plots.
5. DALI standard ImageNet, 1024 tuning, and backend crossover studies.
6. Prepared-input ceilings.
7. DDP DataLoader candidate follow-up and input ceilings.

The plots tell the DataLoader story; the study pages remain the evidence record.
